# 02 - Baseline 1: Vanilla CNN (From Scratch)
**Đồ án:** Chẩn đoán bệnh trên lá cây nông nghiệp

### Nguyên lý thiết kế hệ thống:
- **Designing Machine Learning Systems (Chip Huyen)**: Chương 6 (Start with a Simple Baseline), Chương 7 (Model Size & Latency Trade-off).
- **Hands-On Machine Learning (Aurélien Géron)**: Chương 14 (Deep Computer Vision with CNNs).

### Mục tiêu của Baseline 1:
1. Xác lập mốc đo lường cơ sở (**Macro F1-Score Baseline**).
2. Đo đạc các thông số hệ thống: **Model Footprint (MB)** và **Inference Latency (ms)**.
3. Kiểm thử đường ống huấn luyện (Training Loop) trước khi áp dụng Transfer Learning phức tạp hơn.

In [ ]:
import torch
import torch.nn as nn
from baseline1_simple_cnn import SimplePlantCNN, calculate_model_footprint
from data_pipeline import build_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Khởi tạo mô hình
model = SimplePlantCNN(num_classes=38).to(device)
num_params, size_mb = calculate_model_footprint(model)
print(f"Tổng số tham số: {num_params:,}")
print(f"Dung lượng tham số ước tính: {size_mb:.2f} MB")
print(model)

### 2. Tải Dữ liệu & Khởi tạo Loss Function có Class Weights

In [ ]:
train_loader, valid_loader, idx_to_class, class_weights = build_dataloaders(
    data_dir="../data",
    batch_size=64,
    image_size=224,
    num_workers=2
)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
print("Pipeline & Optimizer sẵn sàng!")

### 3. Huấn Luyện & Đánh Giá Macro F1 cùng Độ Trễ (Inference Latency)

In [ ]:
from baseline1_simple_cnn import train_one_epoch, evaluate

# Chạy thử nghiệm 1 epoch để quan sát các chỉ số
train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
val_metrics = evaluate(model, valid_loader, criterion, device)

print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
print(f"Val Loss: {val_metrics['val_loss']:.4f} | Val Acc: {val_metrics['val_acc']*100:.2f}%")
print(f"Val Macro F1: {val_metrics['val_macro_f1']:.4f} | Val Weighted F1: {val_metrics['val_weighted_f1']:.4f}")
print(f"Inference Latency: {val_metrics['avg_latency_ms']:.2f} ms/ảnh")